# P126 — Transformer de comprensión de documentos sin OCR

## 1. Título y paper

**Paper:** *OCR-free Document Understanding Transformer*  
**Autoría:** Geewook Kim, Teakgyu Hong, Moonbin Yim, JeongYeon Nam, Jinyoung Park, Jinyeong Yim, Wonseok Hwang, Sangdoo Yun, Dongyoon Han, Seunghyun Park  
**Año y venue:** 2022 · ECCV 2022, 498–517  
**Nivel:** L3 · **Motor:** `donut`  
**Ficha completa:** [`P126_donut`](../../papers/foundational/P126_donut/README.md)

**Hito:** Va de la imagen del documento a la salida estructurada sin pasar por OCR, y con ello elimina una fuente de error que la etapa siguiente no podía corregir.

- [doi:10.1007/978-3-031-19815-1_29](https://doi.org/10.1007/978-3-031-19815-1_29)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La tubería OCR más analizador arrastra dos costes: los errores del OCR llegan intactos al final y se componen carácter a carácter, y el OCR hay que licenciarlo y mantenerlo por idioma.
2. Ejecutar una implementación mínima de la propuesta: Un codificador de imagen y un decodificador que emite directamente la estructura —JSON, pares clave-valor—, preentrenado con la tarea de leer el documento completo. Sin etapa intermedia, no hay error que heredar.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P125
- P46


## 4. Intuición

Un OCR con 1 % de error por carácter suena excelente. Sobre documentos de 54 caracteres, deja solo la mitad de documentos sin un solo error — y el analizador que viene detrás no puede arreglar ninguno.


## 5. Concepto mínimo

```text
Cascada:  imagen ──[OCR]──▶ texto ──[analizador]──▶ estructura

P(documento correcto) = (1 − p)^(caracteres)      ← se COMPONE, no se suma

Sin OCR:  imagen ──────────[modelo]──────────▶ estructura
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('donut', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué fracción de documentos sobrevive a un OCR con 1 % de error?
2. ¿Y con 5 %?
3. ¿Sigue la medición a la fórmula?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('donut', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('donut', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con **1 %** por carácter: **0,905** de campos correctos pero solo **0,55** de documentos enteros. Con **5 %**: **0,053** de documentos. Y la medición sigue a la predicción — para 1 % la fórmula da **0,581** y se mide **0,55**. No es mala suerte: es la aritmética de encadenar etapas.


## 10. Comentario pedagógico

El argumento de prescindir del OCR no es que el modelo de extremo a extremo lea mejor: es que **no hereda errores de una etapa que no puede corregir**, porque esa etapa no existe. A cambio introduce los suyos, que sí son entrenables. Es una decisión sobre dónde quieres que viva el error, no sobre quién lee mejor.


## 11. Error o anti-patrón deliberado

Anti-patrón: fijar el objetivo de calidad del OCR en la tasa por carácter.


In [ ]:
print('Un 99 % por caracter suena a objetivo cumplido.')
print('Sobre documentos de 54 caracteres son 55 % de documentos correctos.')
print('El objetivo tiene que fijarse en la unidad que le importa al negocio: el documento.')

## 12. Corrección

La cascada completa, con la predicción al lado:


In [ ]:
r = run_paper_lab('donut', seed=3)['result']
for fila in r['cascada_ocr_mas_analizador']:
    print(fila)
print('prediccion frente a medicion:')
for fila in r['prediccion_frente_a_medicion']:
    print('  ', fila)

## 13. Desafío guiado

Explica por qué medir en campos y medir en documentos dan conclusiones opuestas sobre el mismo sistema, y cuál corresponde declarar en un contrato de servicio.


In [ ]:
r = run_paper_lab('donut', seed=3)['result']
show(r)

## 14. Desafío autónomo

Estima la tasa de error por carácter de tu OCR y calcula qué fracción de tus documentos sale sin un solo fallo. Compárala con la que asume tu proceso.


## 15. Evidencia de aprendizaje

Guarda las dos cifras y la diferencia entre lo que asumías y lo que sale.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P126_donut/README.md) · evaluación formal: [`assessments/papers/P126_donut.md`](../../assessments/papers/P126_donut.md)


## 16. Cierre

Cierra la ruta de percepción. Lo siguiente es generar medios, donde el problema deja de ser entender y pasa a ser de quién es lo generado.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
